In [1]:
import pandas as pd
import numpy as np

# Load dataset

path=r"C:\machine learning\repository\Dataset for regression.csv"

df=pd.read_csv(path)
df.head()
# ---------------------------
# 1. Convert datetime
# ---------------------------
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'], errors='coerce')

# Target variable (in seconds)
df['delivery_time'] = (df['actual_delivery_time'] - df['created_at']).dt.total_seconds()

# Drop rows with missing target
df = df.dropna(subset=['delivery_time'])

# ---------------------------
# 2. Feature Engineering
# ---------------------------
df['order_hour'] = df['created_at'].dt.hour
df['order_day'] = df['created_at'].dt.dayofweek

# Drop useless columns
df = df.drop(['created_at', 'actual_delivery_time'], axis=1)

# ---------------------------
# 3. Split features
# ---------------------------
X = df.drop('delivery_time', axis=1)
y = df['delivery_time']

# ---------------------------
# 4. Preprocessing
# ---------------------------
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# ---------------------------
# 5. Model (High Accuracy)
# ---------------------------
from sklearn.ensemble import RandomForestRegressor

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        random_state=42
    ))
])

# ---------------------------
# 6. Train/Test Split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# 7. Train model
# ---------------------------
model.fit(X_train, y_train)

# ---------------------------
# 8. Predictions
# ---------------------------
y_pred = model.predict(X_test)

# ---------------------------
# 9. Evaluation
# ---------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

C:\Users\perfect\AppData\Local\Temp\ipykernel_14068\2972319823.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
C:\Users\perfect\AppData\Local\Temp\ipykernel_14068\2972319823.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'], errors='coerce')
C:\Users\perfect\AppData\Local\Temp\ipykernel_14068\2972319823.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to 

MAE: 5018.186227069776
RMSE: 14418.79157458641
R2 Score: 0.0777715688417907
